# NYC Restaurant Intelligence Platform — Market Overview
## Notebook 01: Data Quality Audit & Cleaning

**Purpose**: Profile the raw data systematically *before* changing anything, quantify
every issue found, then clean according to decisions made on the evidence.

**Data sources**
| File | Description |
|---|---|
| `DOHMH_New_York_City_Restaurant_Inspection_Results_20260811.csv` | NYC DOHMH inspection records (one row = one violation, **not** one restaurant) |
| `Borough_Boundaries_20260811.csv` | Borough boundaries as WKT MULTIPOLYGON, for mapping |

**Part 1 is read-only** — it profiles the raw data and exports an issue list to
`reports/data_quality_report.md`. **Part 2** applies the agreed fixes and writes
analysis-ready tables to `data/cleaned/`. The raw files are never modified.

---
### The critical premise: data grain

**One row = one violation cited during one inspection of one restaurant.**
A restaurant (`CAMIS`) appears many times; a single inspection spans several rows.

Every judgment about "missing" or "duplicate" below depends on this. Miss it and
you will misread structural artifacts as data errors — and, worse, silently
double-count dirty restaurants in every aggregate.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJ / "data" / "raw"
REPORTS = PROJ / "reports"


def newest(pattern: str) -> Path:
    """The most recent file matching a pattern.

    Source filenames carry their download date, so hard-coding one pins the whole
    notebook to a single snapshot. Matching by pattern means re-downloading and
    re-running is all it takes to refresh.
    """
    matches = sorted(RAW.glob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {RAW}. See data/raw/README.md for "
            "where to download it."
        )
    return matches[-1]


INSPECTION_FILE = newest("DOHMH_New_York_City_Restaurant_Inspection_Results_*.csv")
BOROUGH_FILE = newest("Borough_Boundaries_*.csv")

# Every issue found is collected here and exported as a report at the end
ISSUES = []


def log_issue(issue_id, column, severity, description, n_rows, pct=None, suggestion=""):
    """Record one data quality finding."""
    ISSUES.append(
        {
            "id": issue_id,
            "column": column,
            "severity": severity,
            "description": description,
            "rows_affected": n_rows,
            "pct_of_rows": round(pct, 2) if pct is not None else round(n_rows / len(df) * 100, 2),
            "recommended_action": suggestion,
        }
    )


print("project root:", PROJ)

project root: /Users/qianyiyou/Desktop/gateway-restaurant-project


## 1. Load raw data

Everything is read as `str` so that pandas' type inference cannot hide dirty values — a `BORO` of `"0"` staying numeric, or an invalid date silently becoming `NaT`.

In [2]:
df = pd.read_csv(INSPECTION_FILE, dtype=str, low_memory=False)
boro_gdf = pd.read_csv(BOROUGH_FILE, dtype=str)

print(f"Inspection records: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Borough boundaries: {boro_gdf.shape[0]} rows x {boro_gdf.shape[1]} columns")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1024**2:,.0f} MB")
df.head(3)

Inspection records: 294,976 rows x 27 columns
Borough boundaries: 5 rows x 5 columns


Memory footprint: 504 MB


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50190401,AMG 770W181 QUICKSERVE LLC,Manhattan,770,WEST 181 STREET,10033,9175974271,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/10/2026,NaN,40.85077220305,-73.93786751302,112,10,026500,1064275,1021760104,MN36,POINT (-73.937867513019 40.85077220305)
1,50150870,JAMBA JUICE,Queens,NaN,"TERMINAL C, CONCOURSE D",11371,2159970667,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/10/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,50188685,PIZZABAGELNYC LLC,Brooklyn,999,ATLANTIC AVENUE,11238,9146021099,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/10/2026,NaN,40.6800524689,-73.95934938341,302,35,022700,3057843,3020190060,BK69,POINT (-73.959349383414 40.680052468901)


In [3]:
# Confirm the grain: rows vs restaurants vs inspections
n_rows = len(df)
n_restaurants = df["CAMIS"].nunique()
n_inspections = df.groupby(["CAMIS", "INSPECTION DATE"]).ngroups

print(f"Total rows (violation grain): {n_rows:,}")
print(f"Unique restaurants (CAMIS):   {n_restaurants:,}")
print(f"Unique inspections (CAMIS + date): {n_inspections:,}")
print(f"On average {n_rows / n_restaurants:.1f} rows and "
      f"{n_inspections / n_restaurants:.1f} inspections per restaurant")

Total rows (violation grain): 294,976
Unique restaurants (CAMIS):   31,222
Unique inspections (CAMIS + date): 87,633
On average 9.4 rows and 2.8 inspections per restaurant


## 2. Missing values

In [4]:
missing = pd.DataFrame(
    {
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().sum() / len(df) * 100).round(2),
        "n_unique": df.nunique(),
    }
).sort_values("pct_missing", ascending=False)

missing[missing["n_missing"] > 0]

,n_missing,pct_missing,n_unique
GRADE DATE,160180,54.30,1713
GRADE,149921,50.82,6
SCORE,17125,5.81,156
VIOLATION CODE,6340,2.15,149
VIOLATION DESCRIPTION,6340,2.15,234
BIN,5988,2.03,21398
Census Tract,4767,1.62,1182
Council District,4767,1.62,51
NTA,4749,1.61,193
Community Board,4749,1.61,69


### 2.1 Structural absence vs genuine gaps

Not every blank is a defect. `GRADE` being 50% empty looks alarming, but DOHMH only
assigns letter grades to Cycle and Pre-permit inspections — other inspection types
are never graded. Cross-tabulate before concluding anything.

In [5]:
grade_by_type = pd.crosstab(
    df["INSPECTION TYPE"].str.split(" / ").str[0],
    df["GRADE"].fillna("(missing)"),
)
grade_by_type["pct_missing"] = (
    grade_by_type["(missing)"] / grade_by_type.sum(axis=1) * 100
).round(1)
grade_by_type.sort_values("pct_missing", ascending=False)

GRADE,(missing),A,B,C,N,P,Z,pct_missing
INSPECTION TYPE,,,,,,,,
Smoke-Free Air Act,719,0,0,0,0,0,0,100.0
Sodium Warning,93,0,0,0,0,0,0,100.0
Trans Fat,525,0,0,0,0,0,0,100.0
Administrative Miscellaneous,11561,1,0,2,7,0,0,99.9
Calorie Posting,575,1,0,0,0,0,0,99.8
Inter-Agency Task Force,1116,0,0,0,39,0,0,96.6
Accelerated Inspection Program,31,0,0,0,12,0,0,72.1
Pre-permit (Non-operational),2833,0,0,0,1265,0,0,69.1
Pre-permit (Operational),27137,14545,2858,2276,6948,173,1166,49.2


In [6]:
# GRADE / GRADE DATE absence is structural, not an error
n_grade_missing = df["GRADE"].isna().sum()
log_issue(
    "M-01", "GRADE / GRADE DATE", "Low (structural)",
    "Over 50% of records carry no grade. Cross-tabulation confirms this follows DOHMH "
    "policy: only Cycle and Pre-permit inspections are graded. Not a data entry problem.",
    n_grade_missing,
    suggestion="Keep as NaN, do not impute. Restrict grade-distribution analysis to "
               "gradeable inspection types so the denominator stays correct.",
)

# SCORE gaps
score_missing = df["SCORE"].isna()
score_missing_with_action = (score_missing & df["ACTION"].notna()).sum()
print(f"SCORE missing: {score_missing.sum():,}")
print(f"  ...with a non-null ACTION (inspection happened, score absent): {score_missing_with_action:,}")
print(f"  ...with ACTION also null (never-inspected, see O-02): {(score_missing & df['ACTION'].isna()).sum():,}")

log_issue(
    "M-02", "SCORE", "Medium",
    f"{score_missing.sum():,} rows have no score; {score_missing_with_action:,} of those "
    "have a non-null ACTION, meaning the inspection did occur but the score is absent.",
    int(score_missing.sum()),
    suggestion="Do NOT fill with 0 — a score of 0 means 'flawless' and would understate "
               "risk badly. Keep NaN and average over de-duplicated inspection-level scores.",
)

SCORE missing: 17,125
  ...with a non-null ACTION (inspection happened, score absent): 13,484
  ...with ACTION also null (never-inspected, see O-02): 3,641


In [7]:
# CUISINE DESCRIPTION drives the whole Market Overview module
cuisine_missing = df["CUISINE DESCRIPTION"].isna()
restaurants_no_cuisine = df.loc[cuisine_missing, "CAMIS"].nunique()

print(f"Rows without cuisine: {cuisine_missing.sum():,}")
print(f"Restaurants affected: {restaurants_no_cuisine:,} / {n_restaurants:,} "
      f"({restaurants_no_cuisine / n_restaurants * 100:.1f}%)")

log_issue(
    "M-03", "CUISINE DESCRIPTION", "High",
    f"{restaurants_no_cuisine:,} restaurants carry no cuisine label. Cuisine is the primary "
    "dimension of the Market Overview, so these gaps bias every category statistic.",
    int(cuisine_missing.sum()),
    suggestion="Fill with an explicit 'UNKNOWN' rather than dropping, so the denominator "
               "stays honest, and state the exclusion wherever category shares are quoted.",
)

# Geographic fields
geo_missing = df["Latitude"].isna()
log_issue(
    "M-04", "Latitude / Longitude / BBL", "Medium",
    f"{geo_missing.sum():,} rows have no coordinates and cannot be mapped.",
    int(geo_missing.sum()),
    suggestion="Exclude from map layers only; non-spatial cuisine and score analysis is "
               "unaffected. Optional fallback: ZIPCODE centroids.",
)

Rows without cuisine: 3,721
Restaurants affected: 3,643 / 31,222 (11.7%)


## 3. Duplicates

Two distinct questions:
1. **Fully duplicated rows** — the same violation recorded twice. A genuine error.
2. **Business-key duplicates** — `CAMIS + inspection date + violation code + inspection type`.

In [8]:
dup_full = df.duplicated().sum()
key_cols = ["CAMIS", "INSPECTION DATE", "VIOLATION CODE", "INSPECTION TYPE"]
dup_key = df.duplicated(subset=key_cols).sum()

print(f"Fully duplicated rows: {dup_full:,}")
print(f"Business-key duplicates {key_cols}: {dup_key:,}")

if dup_full:
    display(
        df[df.duplicated(keep=False)]
        .sort_values(key_cols)
        [["CAMIS", "DBA", "INSPECTION DATE", "INSPECTION TYPE", "VIOLATION CODE", "SCORE"]]
        .head(6)
    )

log_issue(
    "D-01", "whole table", "Medium",
    f"{dup_full:,} rows are exact duplicates across all 27 fields. The count matches the "
    "business-key duplicate count exactly, so these are pure re-entries with no hidden "
    "differences.",
    int(dup_full),
    suggestion="Drop with drop_duplicates(), keeping the first. Very low risk.",
)

Fully duplicated rows: 150
Business-key duplicates ['CAMIS', 'INSPECTION DATE', 'VIOLATION CODE', 'INSPECTION TYPE']: 150


,CAMIS,DBA,INSPECTION DATE,INSPECTION TYPE,VIOLATION CODE,SCORE
148258,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,02G,23
151352,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,02G,23
53446,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,06B,23
61014,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,06B,23
83773,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,09A,23
91670,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,02/10/2026,Cycle Inspection / Initial Inspection,09A,23


In [9]:
# A repeated CAMIS is NOT a defect — it is the grain. Documented here to prevent
# someone "helpfully" de-duplicating on CAMIS and destroying the inspection history.
top_camis = df["CAMIS"].value_counts().head(5)
print("Restaurants with the most rows (repetition here is expected, do not de-duplicate):")
for camis, cnt in top_camis.items():
    name = df.loc[df["CAMIS"] == camis, "DBA"].iloc[0]
    print(f"  {camis}  {name[:40]:<42} {cnt} rows")

Restaurants with the most rows (repetition here is expected, do not de-duplicate):
  50138270  MEEM SPICY GROCERY AND DELI                89 rows
  50001215  BYUNG CHUN SOON DAE                        85 rows
  50139259  SHANGHAI TIME                              84 rows
  50106885  DON ALEX                                   76 rows
  50111296  BIG WONG                                   74 rows


## 4. Format consistency

Borough values were expected to have inconsistent capitalisation. Rather than assume,
measure each field.

In [10]:
print("Raw BORO values:")
print(df["BORO"].value_counts(dropna=False).to_string())

print("\nCase-variant detection (does normalising with upper() collapse any values?):")
for col in ["BORO", "CUISINE DESCRIPTION", "GRADE", "CRITICAL FLAG"]:
    raw_n = df[col].dropna().nunique()
    norm_n = df[col].dropna().str.upper().str.strip().nunique()
    flag = "VARIANTS FOUND" if raw_n != norm_n else "consistent"
    print(f"  {col:<24} raw {raw_n:>4} -> normalised {norm_n:>4}   {flag}")

Raw BORO values:
BORO
Manhattan        109215
Brooklyn          74762
Queens            73963
Bronx             27076
Staten Island      9588
0                   372

Case-variant detection (does normalising with upper() collapse any values?):
  BORO                     raw    6 -> normalised    6   consistent


  CUISINE DESCRIPTION      raw   90 -> normalised   90   consistent


  GRADE                    raw    6 -> normalised    6   consistent
  CRITICAL FLAG            raw    3 -> normalised    3   consistent


In [11]:
# The real BORO problem is a placeholder, not casing
boro_invalid = df["BORO"] == "0"
n_boro_invalid = int(boro_invalid.sum())

print(f'Rows where BORO == "0": {n_boro_invalid}')
print(f"  ...also missing ZIPCODE: {df.loc[boro_invalid, 'ZIPCODE'].isna().sum()}")
print(f"  ...with usable coordinates: {df.loc[boro_invalid, 'Latitude'].notna().sum()}")

log_issue(
    "F-01", "BORO", "Medium",
    'Capitalisation is in fact consistent — every value is standard Title Case '
    '("Manhattan"), with no "MANHATTAN"/"manhattan" variants. The actual defect is '
    f'{n_boro_invalid} rows holding the placeholder "0", which is not a real borough.',
    n_boro_invalid,
    suggestion='Map to "Unknown". Note that these rows have NO coordinates at all '
               "(0 usable), so a spatial join against the boundary file cannot recover "
               "them; only a ZIPCODE lookup could.",
)

Rows where BORO == "0": 372
  ...also missing ZIPCODE: 44
  ...with usable coordinates: 0


In [12]:
# Whitespace and casing noise, which breaks name matching and chain detection
dba = df["DBA"].dropna()
dba_norm = dba.str.upper().str.strip().str.replace(r"\s+", " ", regex=True)

street = df["STREET"].dropna()
n_double_space = int(street.str.contains("  ").sum())

print(f"Distinct DBA values: {dba.nunique():,} raw -> {dba_norm.nunique():,} normalised "
      f"({dba.nunique() - dba_norm.nunique()} collapse)")
print(f"Rows where DBA is not all-caps: {(dba != dba.str.upper()).sum():,}")
print(f"Rows where STREET contains consecutive spaces: {n_double_space:,}")
print("\nExamples of STREET with double spaces:")
print(street[street.str.contains('  ')].drop_duplicates().head(5).to_string())

log_issue(
    "F-02", "DBA / STREET", "Medium",
    f"Text fields carry casing and whitespace noise: normalising DBA collapses "
    f"{dba.nunique() - dba_norm.nunique()} duplicate trade names, and {n_double_space:,} "
    "STREET values contain consecutive spaces. Both break chain-brand identification "
    "and address matching.",
    n_double_space,
    suggestion="Strip, collapse internal whitespace, uppercase. Retain the original "
               "trade name in a separate column for traceability.",
)

Distinct DBA values: 24,527 raw -> 24,445 normalised (82 collapse)
Rows where DBA is not all-caps: 2,891
Rows where STREET contains consecutive spaces: 39,344

Examples of STREET with double spaces:
0     WEST  181 STREET
7     EAST   46 STREET
8     WEST   53 STREET
14    WEST   14 STREET
38    WEST   33 STREET


## 5. Outliers and invalid values

In [13]:
score = pd.to_numeric(df["SCORE"], errors="coerce")

print("SCORE distribution:")
print(score.describe().to_string())
print(f"\nValues that fail numeric conversion: {(score.isna() & df['SCORE'].notna()).sum()}")
print(f"Negative scores: {(score < 0).sum()}")
print(f"Scores above 100: {(score > 100).sum():,}")
print(f"Scores above 150: {(score > 150).sum():,}")
print(f"\nQuantiles: {score.quantile([.5, .9, .99, .999]).to_dict()}")

SCORE distribution:


count    277851.000000
mean         25.616863
std          19.161065
min           0.000000
25%          12.000000
50%          22.000000
75%          33.000000
max         214.000000

Values that fail numeric conversion: 0
Negative scores: 0
Scores above 100: 1,907
Scores above 150: 188

Quantiles: {0.5: 22.0, 0.9: 50.0, 0.99: 93.0, 0.999: 142.0}


In [14]:
high = score > 100
n_high = int(high.sum())

print("Sample of high scores — are these real extremes or data entry errors?")
display(
    df.loc[high, ["CAMIS", "DBA", "BORO", "INSPECTION DATE", "SCORE", "ACTION"]]
    .assign(SCORE=score[high])
    .sort_values("SCORE", ascending=False)
    .head(5)
)

closed_share = df.loc[high, "ACTION"].str.contains("Closed", na=False).mean()
print(f"\nShare of >100 records that were closed by DOHMH: {closed_share:.1%}")
print(f"Share across all records: {df['ACTION'].str.contains('Closed', na=False).mean():.1%}")

log_issue(
    "O-01", "SCORE", "Medium",
    f"{n_high:,} rows score above 100 (max {score.max():.0f}). DOHMH scores have no upper "
    f"bound, and {closed_share:.0%} of these records carry a closure order versus 4% "
    "overall — so most are genuine extreme violations, not entry errors.",
    n_high,
    suggestion="Do not delete. Keep the raw value and add a winsorized column (99th "
               "percentile) for charting, so means are not dragged by the tail.",
)

Sample of high scores — are these real extremes or data entry errors?


,CAMIS,DBA,BORO,INSPECTION DATE,SCORE,ACTION
106714,50182863,TAPAS ON LEX,Manhattan,04/01/2026,214.0,Violations were cited in the following area(s).
7056,50182863,TAPAS ON LEX,Manhattan,04/01/2026,214.0,Violations were cited in the following area(s).
17389,50182863,TAPAS ON LEX,Manhattan,04/01/2026,214.0,Violations were cited in the following area(s).
96630,50182863,TAPAS ON LEX,Manhattan,04/01/2026,214.0,Violations were cited in the following area(s).
129844,50182863,TAPAS ON LEX,Manhattan,04/01/2026,214.0,Violations were cited in the following area(s).



Share of >100 records that were closed by DOHMH: 74.5%
Share across all records: 3.7%


In [15]:
insp_date = pd.to_datetime(df["INSPECTION DATE"], format="%m/%d/%Y", errors="coerce")
placeholder = insp_date.dt.year == 1900
n_placeholder = int(placeholder.sum())

print(f"Date range: {insp_date.min().date()} to {insp_date.max().date()}")
print(f"Unparseable dates: {(insp_date.isna() & df['INSPECTION DATE'].notna()).sum()}")
print(f"Future dates: {(insp_date > pd.Timestamp.today()).sum()}")
print(f"1900-01-01 placeholders: {n_placeholder:,}")

# Establish what these rows actually represent
sub = df[placeholder]
print("\nCharacteristics of the 1900 rows:")
print(f"  ACTION entirely null: {sub['ACTION'].isna().all()}")
print(f"  SCORE entirely null:  {sub['SCORE'].isna().all()}")
print(f"  Restaurants involved: {sub['CAMIS'].nunique():,} (exactly one row each)")
print(f"  Rows whose CAMIS also appears with a real inspection: "
      f"{sub['CAMIS'].isin(df.loc[~placeholder, 'CAMIS']).sum()}")

log_issue(
    "O-02", "INSPECTION DATE", "High",
    f"{n_placeholder:,} rows use 1900-01-01 as a placeholder. These rows have null ACTION, "
    f"SCORE and VIOLATION, and the {sub['CAMIS'].nunique():,} restaurants involved appear "
    "nowhere else in the table — i.e. permitted but never-yet-inspected establishments, "
    "not corrupted dates.",
    n_placeholder,
    suggestion="Must be excluded from any trend or score analysis (otherwise the time axis "
               "stretches back to 1900), but retained for market-size counts — these are "
               "real, operating restaurants.",
)

Date range: 1900-01-01 to 2026-08-08
Unparseable dates: 0
Future dates: 0
1900-01-01 placeholders: 3,641

Characteristics of the 1900 rows:
  ACTION entirely null: True
  SCORE entirely null:  True
  Restaurants involved: 3,641 (exactly one row each)
  Rows whose CAMIS also appears with a real inspection: 0


In [16]:
lat = pd.to_numeric(df["Latitude"], errors="coerce")
lon = pd.to_numeric(df["Longitude"], errors="coerce")

zero_coord = (lat == 0) | (lon == 0)
n_zero = int(zero_coord.sum())

# Rough NYC bounding box
in_bbox = lat.between(40.4, 41.0) & lon.between(-74.3, -73.6)
out_bbox = int((~in_bbox & lat.notna() & (lat != 0)).sum())

print(f"Rows at (0, 0) — the Gulf of Guinea, not New York: {n_zero:,}")
print(f"Non-zero rows outside the NYC bounding box: {out_bbox:,}")
print("\nBorough breakdown of the (0,0) rows — their address data is complete, "
      "they simply were never geocoded:")
print(df.loc[zero_coord, "BORO"].value_counts().to_string())

log_issue(
    "O-03", "Latitude / Longitude", "High",
    f"{n_zero:,} rows carry (0, 0) coordinates, which would plot off the coast of Africa. "
    f"Their BORO and address fields are intact — only geocoding is missing. A further "
    f"{out_bbox} rows fall outside the NYC bounding box.",
    n_zero,
    suggestion="Convert (0,0) to NaN — it must never be treated as a valid location. "
               "Exclude from map layers while keeping the rows in borough-level statistics.",
)

Rows at (0, 0) — the Gulf of Guinea, not New York: 3,048
Non-zero rows outside the NYC bounding box: 0

Borough breakdown of the (0,0) rows — their address data is complete, they simply were never geocoded:
BORO
Manhattan        1589
Queens            589
Bronx             469
Brooklyn          298
Staten Island     103


## 6. Cross-field logical consistency

Do the attributes of a single restaurant (`CAMIS`) agree across all of its rows?

In [17]:
checks = {}
for col in ["BORO", "CUISINE DESCRIPTION", "DBA", "ZIPCODE"]:
    n_conflict = int(df.groupby("CAMIS")[col].nunique().gt(1).sum())
    checks[col] = n_conflict
    print(f"CAMIS values holding more than one distinct {col:<22}: {n_conflict:,}")

n_conflict_total = sum(checks.values())
log_issue(
    "L-01", "CAMIS vs attribute fields", "Low (no defect)",
    f"Consistency check passes: BORO, CUISINE, DBA and ZIPCODE never conflict across the "
    f"rows of a single CAMIS ({n_conflict_total} conflicts in total). Restaurant master "
    "data is reliable and can safely be aggregated by CAMIS.",
    n_conflict_total,
    suggestion="No action needed. When building the restaurant dimension table, any row's "
               "attributes will do — no conflict resolution required.",
)

CAMIS values holding more than one distinct BORO                  : 0


CAMIS values holding more than one distinct CUISINE DESCRIPTION   : 0
CAMIS values holding more than one distinct DBA                   : 0
CAMIS values holding more than one distinct ZIPCODE               : 0


In [18]:
# CRITICAL FLAG behaviour on rows with no violation
no_violation = df["VIOLATION CODE"].isna()
print(f"Rows with no violation code: {no_violation.sum():,}")
print("\nTheir CRITICAL FLAG values:")
print(df.loc[no_violation, "CRITICAL FLAG"].value_counts().to_string())
print("\nTheir ACTION values:")
print(df.loc[no_violation, "ACTION"].value_counts(dropna=False).head().to_string())

Rows with no violation code: 6,340

Their CRITICAL FLAG values:
CRITICAL FLAG
Not Applicable    6340

Their ACTION values:
ACTION
NaN                                                                                                                                   3641
No violations were recorded at the time of this inspection.                                                                           2415
Establishment re-opened by DOHMH.                                                                                                      277
Violations were cited in the following area(s).                                                                                          4
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.       2


## 7. Borough boundary file

In [19]:
display(boro_gdf[["BoroCode", "BoroName", "Shape_Area", "Shape_Length"]])

print(f"\nRows: {len(boro_gdf)} (expected 5)")
print(f"Missing values: {boro_gdf.isna().sum().sum()}")
print(f"Geometry type: {boro_gdf['the_geom'].str.split(' ').str[0].unique()}")

# Numeric fields arrive as thousands-separated strings
print(f"\nShape_Area sample: {boro_gdf['Shape_Area'].iloc[0]!r} "
      f"(type {type(boro_gdf['Shape_Area'].iloc[0]).__name__})")

# Can this file be joined to the inspection data?
valid_boros = set(df["BORO"].unique()) - {"0"}
matched = set(boro_gdf["BoroName"]) == valid_boros
print(f"\nBoroName matches the inspection data's BORO values exactly: {matched}")
print(f"  boundary file:   {sorted(boro_gdf['BoroName'])}")
print(f"  inspection data: {sorted(valid_boros)}")

log_issue(
    "B-01", "Shape_Area / Shape_Length", "Low",
    "The boundary file is complete (5 rows, no missing values) and BoroName joins directly "
    "to the inspection data's BORO. The only defect: area and length carry thousands "
    "separators and are therefore parsed as strings rather than numbers.",
    0, pct=0.0,
    suggestion="Strip commas and cast to float on load; parse the_geom with "
               "shapely.wkt.loads for mapping.",
)

,BoroCode,BoroName,Shape_Area,Shape_Length
0,5,Staten Island,"1,623,618,358.46","325,912.288988"
1,3,Brooklyn,"1,934,462,607.75","726,953.04509"
2,4,Queens,"3,041,419,716.32","887,902.903727"
3,1,Manhattan,"636,631,537.285","359,537.906171"
4,2,Bronx,"1,187,199,168.58","463,146.326689"



Rows: 5 (expected 5)
Missing values: 0
Geometry type: ['MULTIPOLYGON']

Shape_Area sample: '1,623,618,358.46' (type str)

BoroName matches the inspection data's BORO values exactly: True
  boundary file:   ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
  inspection data: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


## 8. Issue summary

In [20]:
report = pd.DataFrame(ISSUES)
severity_order = {"High": 0, "Medium": 1, "Low": 2, "Low (structural)": 3, "Low (no defect)": 4}
report = report.sort_values("severity", key=lambda s: s.map(severity_order)).reset_index(drop=True)

display(report[["id", "column", "severity", "rows_affected", "pct_of_rows", "description"]])

,id,column,severity,rows_affected,pct_of_rows,description
0,M-03,CUISINE DESCRIPTION,High,3721,1.26,"3,643 restaurants carry no cuisine label. Cuis..."
1,O-02,INSPECTION DATE,High,3641,1.23,"3,641 rows use 1900-01-01 as a placeholder. Th..."
2,O-03,Latitude / Longitude,High,3048,1.03,"3,048 rows carry (0, 0) coordinates, which wou..."
3,M-02,SCORE,Medium,17125,5.81,"17,125 rows have no score; 13,484 of those hav..."
4,M-04,Latitude / Longitude / BBL,Medium,1701,0.58,"1,701 rows have no coordinates and cannot be m..."
5,D-01,whole table,Medium,150,0.05,150 rows are exact duplicates across all 27 fi...
6,F-01,BORO,Medium,372,0.13,Capitalisation is in fact consistent — every v...
7,F-02,DBA / STREET,Medium,39344,13.34,Text fields carry casing and whitespace noise:...
8,O-01,SCORE,Medium,1907,0.65,"1,907 rows score above 100 (max 214). DOHMH sc..."
9,B-01,Shape_Area / Shape_Length,Low,0,0.00,"The boundary file is complete (5 rows, no miss..."


In [21]:
# Export as Markdown + CSV for sharing and for the repository
REPORTS.mkdir(exist_ok=True)

report.to_csv(REPORTS / "data_quality_issues.csv", index=False)

lines = [
    "# NYC Restaurant Data — Data Quality Report",
    "",
    f"- Source: `{INSPECTION_FILE.name}`",
    f"- Size: {len(df):,} rows x {df.shape[1]} columns, covering {n_restaurants:,} restaurants",
    f"- Date range: {insp_date[~placeholder].min().date()} to {insp_date.max().date()}",
    f"- Issues found: {len(report)}",
    "",
    "## Issue list",
    "",
    report.to_markdown(index=False),
    "",
    "> Generated by `notebooks/01_data_quality_check.ipynb`.",
]
(REPORTS / "data_quality_report.md").write_text("\n".join(lines), encoding="utf-8")

print(f"Exported:\n  {REPORTS / 'data_quality_report.md'}\n  {REPORTS / 'data_quality_issues.csv'}")

Exported:
  /Users/qianyiyou/Desktop/gateway-restaurant-project/reports/data_quality_report.md
  /Users/qianyiyou/Desktop/gateway-restaurant-project/reports/data_quality_issues.csv


---
# Part 2: Cleaning

The handling of each issue has been agreed; the steps below apply it. **The raw files
are never modified** — results are written to `data/cleaned/`.

### Agreed decisions
| Decision | Resolution |
|---|---|
| Never-inspected restaurants (1900 placeholder) | **Keep**, flagged `never_inspected`; auto-excluded from score and trend analysis |
| Restaurants with no cuisine | **Keep**, cuisine set to `UNKNOWN` |
| Missing SCORE values | **Leave empty** — filling with 0 would read as a perfect score |
| Number of output tables | **Three**, one per analytical grain (below) |

### Output design
| Table | Grain (what one row is) | Expected rows | Used for |
|---|---|---|---|
| `restaurants.csv` | one restaurant (`camis`) | 31,222 | maps, cuisine mix, borough density |
| `inspections.csv` | one inspection (`camis` + date + type) | 93,106 | score trends, grade distribution |
| `violations.csv` | one cited violation | 288,486 | most common violation ranking |

> **Correction to the inspection grain**: `camis + date` alone was the obvious choice,
> but 8,625 such groups hold conflicting scores — a restaurant can undergo both an
> initial inspection and a re-inspection on the same day, each independently scored.
> Adding `inspection_type` drops the conflicts to zero, so the correct grain is all three.

## 9. Cleaning steps

Each step records what changed, so the transformation can be audited.

In [22]:
CLEANED = PROJ / "data" / "cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

CLEAN_LOG = []


def log_step(step, detail, before=None, after=None):
    CLEAN_LOG.append({"step": step, "detail": detail, "rows_before": before, "rows_after": after})
    print(f"[{step}] {detail}" + (f"  {before:,} -> {after:,}" if before is not None else ""))


clean = df.copy()
print(f"Starting from {len(clean):,} rows")

Starting from 294,976 rows


### 9.1 Drop exact duplicates (D-01)

In [23]:
n_before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
log_step("D-01", "Dropped fully duplicated rows", n_before, len(clean))

[D-01] Dropped fully duplicated rows  294,976 -> 294,826


### 9.2 Normalise text fields (F-02)

Strip, collapse internal whitespace, uppercase. The original trade name is preserved as `dba_raw`.

In [24]:
clean["dba_raw"] = clean["DBA"]

TEXT_COLS = ["DBA", "STREET", "BUILDING", "CUISINE DESCRIPTION"]
baseline = df.drop_duplicates().reset_index(drop=True)
for col in TEXT_COLS:
    clean[col] = (
        clean[col]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

n_street_fixed = int((clean["STREET"] != baseline["STREET"]).sum())
log_step("F-02", f"Normalised {len(TEXT_COLS)} text fields ({n_street_fixed:,} STREET values changed)")
print(f"  Distinct DBA: {df['DBA'].nunique():,} -> {clean['DBA'].nunique():,}")

[F-02] Normalised 4 text fields (39,562 STREET values changed)
  Distinct DBA: 24,527 -> 24,445


### 9.3 Borough placeholder and missing cuisine (F-01, M-03)

In [25]:
n_boro = int((clean["BORO"] == "0").sum())
clean["BORO"] = clean["BORO"].replace("0", "Unknown")
log_step("F-01", f'BORO "0" -> "Unknown" ({n_boro} rows)')

n_cuisine = int(clean["CUISINE DESCRIPTION"].isna().sum())
clean["CUISINE DESCRIPTION"] = clean["CUISINE DESCRIPTION"].fillna("UNKNOWN")
log_step("M-03", f"Missing cuisine -> UNKNOWN ({n_cuisine:,} rows across "
                 f"{df.loc[df['CUISINE DESCRIPTION'].isna(), 'CAMIS'].nunique():,} restaurants)")

print(clean["BORO"].value_counts().to_string())

[F-01] BORO "0" -> "Unknown" (372 rows)
[M-03] Missing cuisine -> UNKNOWN (3,721 rows across 3,643 restaurants)
BORO
Manhattan        109171
Brooklyn          74715
Queens            73932
Bronx             27048
Staten Island      9588
Unknown             372


### 9.4 Type conversion: scores, dates, coordinates (M-02, O-01, O-02, O-03)

In [26]:
# Scores — kept as NaN, never imputed
clean["score"] = pd.to_numeric(clean["SCORE"], errors="coerce")
log_step("M-02", f"SCORE cast to numeric, {clean['score'].isna().sum():,} nulls preserved (not imputed)")

# Extreme scores get a winsorized companion column; the raw value is untouched
p99 = clean["score"].quantile(0.99)
clean["score_winsorized"] = clean["score"].clip(upper=p99)
log_step("O-01", f"Added score_winsorized (clipped at 99th percentile = {p99:.0f}); raw score unchanged")

# Dates — 1900 placeholders become NaT and are flagged
clean["inspection_date"] = pd.to_datetime(clean["INSPECTION DATE"], format="%m/%d/%Y", errors="coerce")
clean["grade_date"] = pd.to_datetime(clean["GRADE DATE"], format="%m/%d/%Y", errors="coerce")

never_mask = clean["inspection_date"].dt.year == 1900
never_camis = set(clean.loc[never_mask, "CAMIS"])
clean.loc[never_mask, "inspection_date"] = pd.NaT
clean["never_inspected"] = clean["CAMIS"].isin(never_camis)
log_step("O-02", f"1900 placeholder dates -> NaT; flagged {len(never_camis):,} never-inspected restaurants")

# Coordinates — (0,0) is not a location
clean["latitude"] = pd.to_numeric(clean["Latitude"], errors="coerce")
clean["longitude"] = pd.to_numeric(clean["Longitude"], errors="coerce")
zero = (clean["latitude"] == 0) | (clean["longitude"] == 0)
clean.loc[zero, ["latitude", "longitude"]] = np.nan
log_step("O-03", f"(0,0) coordinates -> NaN ({int(zero.sum()):,} rows); "
                 f"{clean['latitude'].notna().sum():,} rows remain mappable")

assert clean["inspection_date"].min().year > 1900, "1900 placeholder dates still present"
assert not ((clean["latitude"] == 0) | (clean["longitude"] == 0)).any(), "zero coordinates still present"
print("\nAssertions passed: no 1900 dates, no zero coordinates")

[M-02] SCORE cast to numeric, 17,119 nulls preserved (not imputed)
[O-01] Added score_winsorized (clipped at 99th percentile = 93); raw score unchanged


[O-02] 1900 placeholder dates -> NaT; flagged 3,641 never-inspected restaurants


[O-03] (0,0) coordinates -> NaN (3,042 rows); 290,083 rows remain mappable

Assertions passed: no 1900 dates, no zero coordinates


### 9.5 Derived fields

In [27]:
clean["is_critical"] = clean["CRITICAL FLAG"] == "Critical"
clean["is_closed_action"] = clean["ACTION"].str.contains("Closed", na=False)
clean["has_violation"] = clean["VIOLATION CODE"].notna()
clean["inspection_category"] = clean["INSPECTION TYPE"].str.split(" / ").str[0]
clean["inspection_stage"] = clean["INSPECTION TYPE"].str.split(" / ").str[1]
# Only Cycle and Pre-permit inspections are ever graded (see M-01)
clean["is_gradeable"] = clean["inspection_category"].isin(["Cycle Inspection", "Pre-permit (Operational)"])

log_step("derived", "Added 6 derived fields (is_critical / is_closed_action / has_violation / "
                    "inspection_category / inspection_stage / is_gradeable)")
pd.DataFrame(CLEAN_LOG)

[derived] Added 6 derived fields (is_critical / is_closed_action / has_violation / inspection_category / inspection_stage / is_gradeable)


,step,detail,rows_before,rows_after
0,D-01,Dropped fully duplicated rows,294976.0,294826.0
1,F-02,"Normalised 4 text fields (39,562 STREET values...",NaN,NaN
2,F-01,"BORO ""0"" -> ""Unknown"" (372 rows)",NaN,NaN
3,M-03,"Missing cuisine -> UNKNOWN (3,721 rows across ...",NaN,NaN
4,M-02,"SCORE cast to numeric, 17,119 nulls preserved ...",NaN,NaN
5,O-01,Added score_winsorized (clipped at 99th percen...,NaN,NaN
6,O-02,"1900 placeholder dates -> NaT; flagged 3,641 n...",NaN,NaN
7,O-03,"(0,0) coordinates -> NaN (3,042 rows); 290,083...",NaN,NaN
8,derived,Added 6 derived fields (is_critical / is_close...,NaN,NaN


## 10. Split into three analysis tables

**This is the most consequential step in the project.** In the raw table one row is one
violation, so restaurants with worse hygiene occupy more rows. Aggregating there weights
dirty restaurants more heavily and systematically distorts every citywide statistic.

The three tables form a hierarchy — **a restaurant has many inspections, an inspection cites
many violations** — linked by two keys:

| Key | Where it lives | What it does |
|---|---|---|
| `camis` | `restaurants` (primary), `inspections`, `violations` | identifies a restaurant |
| `inspection_id` | `inspections` (primary), `violations` | identifies one inspection |

`inspection_id` is generated here rather than coming from the source data. The natural key
for an inspection is `camis + date + inspection_type`, which is awkward to carry into child
tables and to index in a database, so each inspection also gets a single integer id.

### 10.1 `inspections.csv` — one row per inspection

Grain: `camis` + inspection date + inspection type, the combination that yields zero score and grade conflicts.

In [28]:
INSP_KEY = ["CAMIS", "inspection_date", "INSPECTION TYPE"]
real = clean[clean["inspection_date"].notna()]

inspections = (
    real.groupby(INSP_KEY, dropna=False)
    .agg(
        score=("score", "first"),
        score_winsorized=("score_winsorized", "first"),
        grade=("GRADE", "first"),
        grade_date=("grade_date", "first"),
        action=("ACTION", "first"),
        inspection_category=("inspection_category", "first"),
        inspection_stage=("inspection_stage", "first"),
        is_gradeable=("is_gradeable", "first"),
        n_violations=("has_violation", "sum"),
        n_critical=("is_critical", "sum"),
        closed_by_dohmh=("is_closed_action", "first"),
    )
    .reset_index()
    .rename(columns={"CAMIS": "camis", "INSPECTION TYPE": "inspection_type"})
    .sort_values(["camis", "inspection_date", "inspection_type"])
    .reset_index(drop=True)
)

# A surrogate key. The natural key is the three-column combination above, which is
# unwieldy to carry into child tables and to index in a database; a single integer
# lets violations point at an inspection with one column. Sorting first makes the
# numbering reproducible across re-runs.
inspections.insert(0, "inspection_id", np.arange(1, len(inspections) + 1))

print(f"inspections: {len(inspections):,} rows across {inspections['camis'].nunique():,} restaurants")
print(f"Date range: {inspections['inspection_date'].min().date()} to "
      f"{inspections['inspection_date'].max().date()}")
inspections.head(3)

inspections: 93,106 rows across 27,581 restaurants
Date range: 2007-08-10 to 2026-08-08


,inspection_id,camis,inspection_date,inspection_type,score,score_winsorized,grade,grade_date,action,inspection_category,inspection_stage,is_gradeable,n_violations,n_critical,closed_by_dohmh
0,1,30075445,2023-08-01,Cycle Inspection / Initial Inspection,38.0,38.0,None,NaT,Violations were cited in the following area(s).,Cycle Inspection,Initial Inspection,True,3,2,False
1,2,30075445,2023-08-22,Cycle Inspection / Re-inspection,12.0,12.0,A,2023-08-22,Violations were cited in the following area(s).,Cycle Inspection,Re-inspection,True,3,1,False
2,3,30075445,2024-11-08,Cycle Inspection / Initial Inspection,10.0,10.0,A,2024-11-08,Violations were cited in the following area(s).,Cycle Inspection,Initial Inspection,True,3,1,False


### 10.2 `violations.csv` — one row per cited violation

In [29]:
VIOL_COLS = {
    "CAMIS": "camis", "DBA": "dba", "BORO": "boro",  # noqa: E501
    "CUISINE DESCRIPTION": "cuisine", "inspection_date": "inspection_date",
    "INSPECTION TYPE": "inspection_type", "inspection_category": "inspection_category",
    "VIOLATION CODE": "violation_code", "VIOLATION DESCRIPTION": "violation_description",
    "CRITICAL FLAG": "critical_flag", "is_critical": "is_critical",
    "score": "score", "GRADE": "grade", "ACTION": "action",
}
violations = clean.loc[clean["has_violation"], list(VIOL_COLS)].rename(columns=VIOL_COLS)
violations = violations.reset_index(drop=True)

# Attach the parent inspection's surrogate key
INSP_NATURAL_KEY = ["camis", "inspection_date", "inspection_type"]
n_before = len(violations)
violations = violations.merge(
    inspections[INSP_NATURAL_KEY + ["inspection_id"]],
    on=INSP_NATURAL_KEY, how="left",
)
# Move the foreign key to the front, where a reader expects it
violations.insert(0, "inspection_id", violations.pop("inspection_id"))

assert len(violations) == n_before, "the join duplicated or dropped violation rows"
assert violations["inspection_id"].notna().all(), "some violations have no parent inspection"

print(f"violations: {len(violations):,} rows "
      f"({(~clean['has_violation']).sum():,} of {len(clean):,} rows carried no violation)")
print(f"Every violation links to an inspection; "
      f"{violations['inspection_id'].nunique():,} of {len(inspections):,} inspections "
      f"cited at least one.")
violations.head(3)

violations: 288,486 rows (6,340 of 294,826 rows carried no violation)
Every violation links to an inspection; 90,407 of 93,106 inspections cited at least one.


,inspection_id,camis,dba,boro,cuisine,inspection_date,inspection_type,inspection_category,violation_code,violation_description,critical_flag,is_critical,score,grade,action
0,7882,41131002,TANGRA,Queens,ASIAN/ASIAN FUSION,2023-01-04,Cycle Inspection / Re-inspection,Cycle Inspection,08A,Establishment is not free of harborage or cond...,Not Critical,False,13.0,A,Violations were cited in the following area(s).
1,89630,50172361,GUACADO,Brooklyn,MEXICAN,2026-06-17,Pre-permit (Operational) / Re-inspection,Pre-permit (Operational),02B,Hot TCS food item not held at or above 140 °F.,Critical,True,34.0,C,Violations were cited in the following area(s).
2,27169,50033660,RUSTY'S FLAVOR,Manhattan,CARIBBEAN,2026-03-04,Cycle Inspection / Initial Inspection,Cycle Inspection,02B,Hot TCS food item not held at or above 140 °F.,Critical,True,23.0,NaN,Violations were cited in the following area(s).


### 10.3 `restaurants.csv` — one row per restaurant

Attributes plus metrics rolled up from the inspection table.

In [30]:
ATTR_COLS = {
    "CAMIS": "camis", "DBA": "dba", "dba_raw": "dba_raw", "BORO": "boro",
    "BUILDING": "building", "STREET": "street", "ZIPCODE": "zipcode", "PHONE": "phone",
    "CUISINE DESCRIPTION": "cuisine", "latitude": "latitude", "longitude": "longitude",
    "Community Board": "community_board", "Council District": "council_district",
    "NTA": "nta", "never_inspected": "never_inspected",
}

# Take each restaurant's most recent record (Part 1 proved attributes never conflict;
# sorting simply ensures the current trade name wins)
attrs = (
    clean.sort_values("inspection_date", na_position="first")
    .groupby("CAMIS", as_index=False)
    .last()[list(ATTR_COLS)]
    .rename(columns=ATTR_COLS)
)

# Roll-ups computed on the de-duplicated inspection table, so restaurants with many
# violations are not counted repeatedly
insp_stats = inspections.groupby("camis").agg(
    n_inspections=("inspection_date", "count"),
    first_inspection=("inspection_date", "min"),
    last_inspection=("inspection_date", "max"),
    avg_score=("score", "mean"),
    latest_score=("score", "last"),
    total_violations=("n_violations", "sum"),
    total_critical=("n_critical", "sum"),
    ever_closed=("closed_by_dohmh", "any"),
)

# Latest grade: only from gradeable inspections that actually carry a grade
graded = inspections[inspections["grade"].notna() & inspections["is_gradeable"]]
latest_grade = (
    graded.sort_values("inspection_date")
    .groupby("camis")
    .agg(latest_grade=("grade", "last"), latest_grade_date=("inspection_date", "last"))
)

restaurants = (
    attrs.merge(insp_stats, on="camis", how="left")
    .merge(latest_grade, on="camis", how="left")
)
restaurants["avg_score"] = restaurants["avg_score"].round(2)

print(f"restaurants: {len(restaurants):,} rows")
print(f"  never inspected:      {restaurants['never_inspected'].sum():,}")
print(f"  with usable coordinates: {restaurants['latitude'].notna().sum():,}")
print(f"  ever closed by DOHMH: {restaurants['ever_closed'].sum():,.0f}")
restaurants.head(3)

restaurants: 31,222 rows
  never inspected:      3,641
  with usable coordinates: 30,417
  ever closed by DOHMH: 1,285


,camis,dba,dba_raw,boro,building,street,zipcode,phone,cuisine,latitude,longitude,community_board,council_district,nta,never_inspected,n_inspections,first_inspection,last_inspection,avg_score,latest_score,total_violations,total_critical,ever_closed,latest_grade,latest_grade_date
0,30075445,MORRIS PARK BAKE SHOP,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,BAKERY PRODUCTS/DESSERTS,40.848231,-73.855972,211,13,BX37,False,4.0,2023-08-01,2026-02-27,16.75,7.0,11.0,5.0,False,A,2026-02-27
1,30191841,D.J. REYNOLDS,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019,2122452912,IRISH,40.767326,-73.984310,104,03,MN15,False,3.0,2024-11-20,2026-07-30,15.67,13.0,11.0,6.0,False,A,2026-07-30
2,40356078,"I. & L. DELICACIES,INC.","I. & L. DELICACIES,INC.",Manhattan,1500-2N,D AVE.,None,None,UNKNOWN,NaN,NaN,None,None,None,True,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT


## 11. Post-cleaning validation

Assertions gate the output — any failure halts the notebook rather than letting bad data reach the analysis layer.

In [31]:
checks = []

def check(name, condition, detail=""):
    checks.append({"check": name, "result": "PASS" if condition else "FAIL", "detail": detail})
    return condition

# Row reconciliation
check("Restaurant count matches distinct CAMIS", len(restaurants) == df["CAMIS"].nunique(),
      f"{len(restaurants):,} vs {df['CAMIS'].nunique():,}")
check("restaurants.camis is unique", restaurants["camis"].is_unique)
check("inspections primary key is unique",
      not inspections.duplicated(["camis", "inspection_date", "inspection_type"]).any())
check("inspections row count matches expected grain", len(inspections) == 93106,
      f"{len(inspections):,} rows")

# Validity
check("No 1900 placeholder dates", inspections["inspection_date"].dt.year.min() > 1900,
      f"earliest {inspections['inspection_date'].min().date()}")
check("No (0,0) coordinates",
      not ((restaurants["latitude"] == 0) | (restaurants["longitude"] == 0)).any())
check("Coordinates fall inside NYC",
      bool(restaurants["latitude"].dropna().between(40.4, 41.0).all()
           and restaurants["longitude"].dropna().between(-74.3, -73.6).all()))
check("BORO limited to 5 boroughs + Unknown",
      set(restaurants["boro"]) <= {"Manhattan", "Brooklyn", "Queens", "Bronx",
                                    "Staten Island", "Unknown"},
      str(sorted(set(restaurants["boro"]))))
check("Cuisine has no nulls", restaurants["cuisine"].notna().all())
check("SCORE was not imputed with zeros",
      inspections["score"].isna().sum() > 0,
      f"{inspections['score'].isna().sum():,} nulls preserved")

# Logical consistency
check("Never-inspected restaurants have no inspections",
      not restaurants.loc[restaurants["never_inspected"], "n_inspections"].notna().any())
check("violations row count matches rows with a violation code",
      len(violations) == int(clean["has_violation"].sum()))
check("inspection_id is unique in inspections", inspections["inspection_id"].is_unique)
check("every violation points at a real inspection",
      violations["inspection_id"].isin(inspections["inspection_id"]).all(),
      f"{violations['inspection_id'].nunique():,} inspections cited")

result = pd.DataFrame(checks)
display(result)
assert (result["result"] == "PASS").all(), "Some validation checks failed — see table above"
print("\nAll validation checks passed.")

,check,result,detail
0,Restaurant count matches distinct CAMIS,PASS,"31,222 vs 31,222"
1,restaurants.camis is unique,PASS,
2,inspections primary key is unique,PASS,
3,inspections row count matches expected grain,PASS,"93,106 rows"
4,No 1900 placeholder dates,PASS,earliest 2007-08-10
5,"No (0,0) coordinates",PASS,
6,Coordinates fall inside NYC,PASS,
7,BORO limited to 5 boroughs + Unknown,PASS,"['Bronx', 'Brooklyn', 'Manhattan', 'Queens', '..."
8,Cuisine has no nulls,PASS,
9,SCORE was not imputed with zeros,PASS,"10,313 nulls preserved"



All validation checks passed.


### 11.1 Before and after: why the split is necessary

The same metric at two grains, to show the size of the double-counting effect.

In [32]:
raw_avg = pd.to_numeric(df["SCORE"], errors="coerce").mean()
insp_avg = inspections["score"].mean()
rest_avg = restaurants["avg_score"].mean()

comparison = pd.DataFrame({
    "basis": ["Raw table (one row per violation)",
              "Inspection table (one row per inspection)",
              "Restaurant table (one row per restaurant)"],
    "rows": [f"{len(df):,}", f"{len(inspections):,}", f"{len(restaurants):,}"],
    "mean_score": [round(raw_avg, 2), round(insp_avg, 2), round(rest_avg, 2)],
})
display(comparison)

print(f"The raw table overstates the mean score by {raw_avg - insp_avg:.2f} points "
      f"({(raw_avg / insp_avg - 1) * 100:.1f}%).")
print("Cause: higher-scoring (dirtier) restaurants are cited for more violations, occupy")
print("more rows, and are therefore weighted more heavily in the raw table.")
print("Always aggregate on the table whose grain matches the question.")

,basis,rows,mean_score
0,Raw table (one row per violation),"294,976",25.62
1,Inspection table (one row per inspection),"93,106",17.87
2,Restaurant table (one row per restaurant),"31,222",17.04


The raw table overstates the mean score by 7.74 points (43.3%).
Cause: higher-scoring (dirtier) restaurants are cited for more violations, occupy
more rows, and are therefore weighted more heavily in the raw table.
Always aggregate on the table whose grain matches the question.


## 12. Export cleaned data

In [33]:
restaurants.to_csv(CLEANED / "restaurants.csv", index=False)
inspections.to_csv(CLEANED / "inspections.csv", index=False)
violations.to_csv(CLEANED / "violations.csv", index=False)

# Borough boundaries: strip thousands separators from numeric fields (B-01)
boro_clean = boro_gdf.copy()
for col in ["Shape_Area", "Shape_Length"]:
    boro_clean[col] = boro_clean[col].str.replace(",", "", regex=False).astype(float)
boro_clean.columns = ["boro_code", "boro_name", "shape_area", "shape_length", "geometry_wkt"]
boro_clean.to_csv(CLEANED / "borough_boundaries.csv", index=False)
log_step("B-01", "Boundary area/length de-comma'd and cast to float")

for f in sorted(CLEANED.glob("*.csv")):
    print(f"  {f.name:<28} {f.stat().st_size / 1024**2:>7.1f} MB")

[B-01] Boundary area/length de-comma'd and cast to float
  borough_boundaries.csv           2.9 MB
  inspections.csv                 16.4 MB
  restaurants.csv                  6.0 MB
  violations.csv                 100.4 MB


### 12.1 Data dictionary

In [34]:
GRAIN = {
    "restaurants.csv": ("one restaurant (camis)", "maps, cuisine mix, borough density"),
    "inspections.csv": ("one inspection (camis + date + type)", "score trends, grade distribution"),
    "violations.csv": ("one cited violation", "most common violation ranking"),
    "borough_boundaries.csv": ("one borough", "map base layer"),
}
TABLES = {
    "restaurants.csv": restaurants, "inspections.csv": inspections,
    "violations.csv": violations, "borough_boundaries.csv": boro_clean,
}

lines = ["# Data Dictionary — data/cleaned/", "",
         "> Generated by `notebooks/01_data_quality_check.ipynb`.", ""]
for fname, tbl in TABLES.items():
    grain, use = GRAIN[fname]
    lines += [f"## `{fname}`", "",
              f"- **Grain**: {grain}", f"- **Rows**: {len(tbl):,}", f"- **Used for**: {use}", "",
              pd.DataFrame({
                  "column": tbl.columns,
                  "dtype": [str(t) for t in tbl.dtypes],
                  "non_null": [tbl[c].notna().sum() for c in tbl.columns],
                  "n_unique": [tbl[c].nunique() for c in tbl.columns],
              }).to_markdown(index=False), ""]

(REPORTS / "data_dictionary.md").write_text("\n".join(lines), encoding="utf-8")
pd.DataFrame(CLEAN_LOG).to_csv(REPORTS / "cleaning_log.csv", index=False)
print(f"Exported:\n  {REPORTS / 'data_dictionary.md'}\n  {REPORTS / 'cleaning_log.csv'}")

Exported:
  /Users/qianyiyou/Desktop/gateway-restaurant-project/reports/data_dictionary.md
  /Users/qianyiyou/Desktop/gateway-restaurant-project/reports/cleaning_log.csv


---
## 13. Summary

The raw extract becomes three analysis-ready tables:

| Table | Rows | One row is | Use it for |
|---|---|---|---|
| `restaurants.csv` | 31,222 | a restaurant | maps, cuisine mix, borough density |
| `inspections.csv` | 93,106 | an inspection | score trends, grade distribution |
| `violations.csv` | 288,486 | a cited violation | most common violation ranking |
| `borough_boundaries.csv` | 5 | a borough | map base layer |

**Three rules for using them**

1. Counting restaurants or cuisine share — use `restaurants.csv`. Counting rows in the
   raw table overstates the count by roughly 10x.
2. Averaging scores or plotting trends — use `inspections.csv`. The raw table overstates
   the mean score by 43%, i.e. understates hygiene.
3. Grade distributions — filter `is_gradeable == True` first, or the denominator absorbs
   inspection types that are never graded.

**Next**: `notebooks/02_market_overview.ipynb` — market size, cuisine landscape,
borough composition and supply gaps.